In [ ]:
LOG_DIR = '../../experiments/current/'


In [ ]:
# Parameters
LOG_DIR = "/Users/yvesb/Documents/Projects/llm-agents-gama/experiments/archive/2026-08-24_17_34/"


In [ ]:
import pandas as pd
import json
import os


def load_jsonl(path):
    """Charge un fichier de JSON objets concaténés (format multi-lignes ou NDJSON)."""
    with open(path) as f:
        content = f.read()
    if not content.strip():
        return pd.DataFrame()
    decoder = json.JSONDecoder()
    records, pos = [], 0
    while pos < len(content):
        try:
            obj, end = decoder.raw_decode(content, pos)
            records.append(obj)
            pos = end
            while pos < len(content) and content[pos] in ' \n\t\r':
                pos += 1
        except json.JSONDecodeError:
            break
    df = pd.DataFrame(records)
    if not df.empty and 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
    return df

# overwrite with actual folder

df_success = load_jsonl(LOG_DIR + 'llm_exchanges.jsonl')
df_error   = load_jsonl(LOG_DIR + 'llm_errors.jsonl')
#df_success = load_jsonl('../../experiments/2026-05-15_12_26/llm_exchanges.jsonl')
#df_error   = load_jsonl('../../experiments/2026-05-15_12_26/llm_errors.jsonl')

print(f"Succès : {len(df_success)} | Erreurs : {len(df_error)}")
df_success.head()

def _img_path(title, ext="png"):
    name = "".join(c if c.isalnum() or c in "-_" else "_" for c in title)
    img_dir = os.path.join(LOG_DIR, "images/llm_traffic")
    os.makedirs(img_dir, exist_ok=True)
    return os.path.join(img_dir, f"{name}.{ext}")


In [ ]:
df_success.columns

df_freq = df_success[['time', 'provider']]

df_freq

## Appels par minute par provider (succès / erreurs)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Combiner succès et erreurs avec statut
df_s = df_success[['time', 'provider']].copy()
df_s['status'] = 'success'

df_e = df_error[['time', 'provider']].copy()
df_e['status'] = 'error'

df_all = pd.concat([df_s, df_e], ignore_index=True)
df_all['minute'] = df_all['time'].dt.floor('min')

# Compter les appels par minute, provider et statut
calls_per_min = (
    df_all.groupby(['minute', 'provider', 'status'])
    .size()
    .unstack('status', fill_value=0)
)

providers_sorted = sorted(df_all['provider'].unique())
n = len(providers_sorted)

fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n), sharex=True)
if n == 1:
    axes = [axes]

for ax, prov in zip(axes, providers_sorted):
    if prov in calls_per_min.index.get_level_values('provider'):
        prov_data = calls_per_min.xs(prov, level='provider')
        if 'success' in prov_data.columns:
            ax.plot(prov_data.index, prov_data['success'], label='succès', color='steelblue', linewidth=1.5)
        if 'error' in prov_data.columns:
            ax.plot(prov_data.index, prov_data['error'], label='erreur', color='tomato', linestyle='--', linewidth=1.5)
    ax.set_title(prov, fontsize=10, fontweight='bold')
    ax.set_ylabel('appels/min')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.grid(True, alpha=0.3)

plt.savefig(_img_path("provider_calls"), dpi=200)

plt.xlabel('Heure')
plt.tight_layout()
plt.show()


## Taux de réussite par provider

In [ ]:
total_calls = df_all.groupby('provider').size().rename('total')
success_counts = df_all[df_all['status'] == 'success'].groupby('provider').size().rename('success')
error_counts = df_all[df_all['status'] == 'error'].groupby('provider').size().rename('errors')

# Max d'appels sur une minute (toutes minutes confondues, success + erreurs)
calls_per_min_total = (
    df_all.groupby(['provider', 'minute'])
    .size()
)
max_calls_per_min = calls_per_min_total.groupby('provider').max().rename('max_appels/min')

summary = pd.concat([total_calls, success_counts, error_counts, max_calls_per_min], axis=1).fillna(0)
summary[['total', 'success', 'errors', 'max_appels/min']] = summary[['total', 'success', 'errors', 'max_appels/min']].astype(int)
summary['taux_réussite'] = (summary['success'] / summary['total'] * 100).round(1)
summary = summary.sort_values('taux_réussite')

# Graphique
fig, ax = plt.subplots(figsize=(10, max(4, len(summary) * 0.7)))

colors = ['green' if v >= 80 else 'orange' if v >= 50 else 'tomato' for v in summary['taux_réussite']]
bars = ax.barh(summary.index, summary['taux_réussite'], color=colors, edgecolor='white')

for bar, val in zip(bars, summary['taux_réussite']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=9)

ax.set_xlabel('Taux de réussite (%)')
ax.set_title('Taux de réussite par provider')
ax.set_xlim(0, 110)
ax.axvline(x=100, color='gray', linestyle='--', alpha=0.4)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(_img_path("provider_success_rate"), dpi=200)
plt.show()

# Tableau récapitulatif
summary.sort_values('taux_réussite', ascending=False)


## Principales erreurs par provider

In [ ]:
summary_errors = pd.DataFrame(
    df_error.groupby(['provider', 'error_type', 'error_message'])
    .size()
    .sort_values(ascending=False)
    .reset_index(name='count')
)

summary_errors = summary_errors[summary_errors['count'] > 1]

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
display(summary_errors.head(20))


## Tokens par catégorie de prompt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Vérification présence colonne category ──────────────────────────────────
if "category" not in df_success.columns or df_success["category"].eq("").all():
    print("⚠️  Colonne 'category' absente ou vide — les échanges loggés avant cet ajout ne contiennent pas la catégorie.")
    print("   Relancez une simulation pour obtenir des données annotées.")
else:
    df_tok = df_success.copy()
    df_tok["tokens_total"] = df_tok["tokens_in"] + df_tok["tokens_out"]
    df_tok["agents_count"] = df_tok["response"].apply(lambda r: max(len(r), 1) if isinstance(r, list) else 1)
    df_tok["tokens_in_per_agent"]  = df_tok["tokens_in"]    / df_tok["agents_count"]
    df_tok["tokens_out_per_agent"] = df_tok["tokens_out"]   / df_tok["agents_count"]

    # ── Stats par catégorie ───────────────────────────────────────────────────
    agg = df_tok.groupby("category").agg(
        appels         =("tokens_in",           "count"),
        agents_total   =("agents_count",         "sum"),
        in_min         =("tokens_in_per_agent",  "min"),
        in_mean        =("tokens_in_per_agent",  "mean"),
        in_max         =("tokens_in_per_agent",  "max"),
        out_min        =("tokens_out_per_agent", "min"),
        out_mean       =("tokens_out_per_agent", "mean"),
        out_max        =("tokens_out_per_agent", "max"),
        total_tokens_in =("tokens_in",  "sum"),
        total_tokens_out=("tokens_out", "sum"),
    ).round(0)

    # inf (division par 0) puis NaN → 0
    bad_cols = agg.columns[agg.isin([np.inf, -np.inf]).any() | agg.isna().any()].tolist()
    if bad_cols:
        print(f"⚠️  Valeurs non-finies (inf/NaN) détectées dans : {bad_cols} — remplacées par 0.")

    agg = (agg
           .replace([np.inf, -np.inf], np.nan)
           .fillna(0)
           .astype({c: int for c in ["appels","agents_total","in_min","in_mean","in_max","out_min","out_mean","out_max","total_tokens_in","total_tokens_out"]}))

    agg["total_tokens_total"] = agg["total_tokens_in"] + agg["total_tokens_out"]

    # ── Totaux session ────────────────────────────────────────────────────────
    session_in    = int(df_tok["tokens_in"].sum())
    session_out   = int(df_tok["tokens_out"].sum())
    session_total = session_in + session_out

    print(f"Session complète — tokens_in={session_in:,}  tokens_out={session_out:,}  total={session_total:,}")
    print()

    # Tableau stats par catégorie
    display(agg[["appels","agents_total",
                 "in_min","in_mean","in_max",
                 "out_min","out_mean","out_max",
                 "total_tokens_in","total_tokens_out","total_tokens_total"]])

    # ── Vérification cohérence avec llm_config.py ────────────────────────────
    ASSUMED_PROMPT_TOKENS = 2200   # Settings.assumed_prompt_tokens
    MIN_OUTPUT_TOKENS     = 512    # Settings.min_output_tokens

    print()
    print("=== Vérification cohérence avec llm_config.py ===")
    any_warning = False
    for cat, row in agg.iterrows():
        flag_in  = row["in_max"]  > ASSUMED_PROMPT_TOKENS
        flag_out = row["out_min"] < MIN_OUTPUT_TOKENS and row["out_mean"] < MIN_OUTPUT_TOKENS
        if flag_in:
            print(f"  ⚠️  [{cat}] tokens_in/agent max={row['in_max']} dépasse assumed_prompt_tokens={ASSUMED_PROMPT_TOKENS}")
            any_warning = True
        if flag_out:
            print(f"  ℹ️  [{cat}] tokens_out/agent mean={row['out_mean']} < min_output_tokens={MIN_OUTPUT_TOKENS} (réponses courtes)")
    if not any_warning:
        print(f"  ✅ tokens_in/agent max ≤ assumed_prompt_tokens={ASSUMED_PROMPT_TOKENS} sur toutes les catégories")

    # ── Graphique min/mean/max tokens_in par catégorie ────────────────────────
    cats = list(agg.index)
    x    = np.arange(len(cats))
    w    = 0.28

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(cats) * 0.9)))

    for ax, col_prefix, label, color in [
        (axes[0], "in",  "tokens_in / agent",  "steelblue"),
        (axes[1], "out", "tokens_out / agent", "darkorange"),
    ]:
        mins   = agg[f"{col_prefix}_min"].values
        means  = agg[f"{col_prefix}_mean"].values
        maxs   = agg[f"{col_prefix}_max"].values

        ax.bar(x - w, mins,  w, label="min",  color=color, alpha=0.5)
        ax.bar(x,     means, w, label="mean", color=color, alpha=0.85)
        ax.bar(x + w, maxs,  w, label="max",  color=color, alpha=1.0)

        if col_prefix == "in":
            ax.axhline(ASSUMED_PROMPT_TOKENS, color="red", linestyle="--", linewidth=1, label=f"assumed_prompt_tokens={ASSUMED_PROMPT_TOKENS}")
        else:
            ax.axhline(MIN_OUTPUT_TOKENS, color="gray", linestyle="--", linewidth=1, label=f"min_output_tokens={MIN_OUTPUT_TOKENS}")

        ax.set_xticks(x)
        ax.set_xticklabels(cats, rotation=20, ha="right")
        ax.set_title(label)
        ax.set_ylabel("tokens")
        ax.legend(fontsize=8)
        ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(_img_path("tokens_par_categorie"), dpi=200)
    plt.show()


## Consommation cumulée de tokens vs budget

Tokens **cumulés depuis le début** (chaque jour = total accumulé), en **aires empilées par catégorie** (ligne noire = total cumulé). La ligne rouge pointillée est le **budget journalier constant = 338 540 000 tokens/jour** (somme des quotas tokens/jour des providers).

> **Échelle standard (linéaire), libellée en M de tokens.** Comme la conso reste très en-dessous du budget journalier, les aires apparaissent quasi plates en bas du graphe — c'est normal, le `print` sous le graphe donne les valeurs exactes en M.

- Axe jour : **jour simulé** (`sim_day`) si présent, sinon repli sur l'horloge murale.
- Si `llm_cache_hits.jsonl` existe, la ligne grise pointillée « sans cache » montre le total cumulé qu'on aurait eu sans le cache (économie valorisée au coût moyen par agent des appels itinary réels).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os, json, re

# Plafond QUOTIDIEN de tokens (somme des quotas tokens/jour des providers) — CONSTANT
DAILY_TOKEN_LIMIT = 338_540_000

df_day = df_success.copy()
df_day["tokens_total"] = df_day["tokens_in"] + df_day["tokens_out"]

# Axe "jour" : on privilégie le jour SIMULÉ (sim_day), sinon repli sur l'horloge murale.
if "sim_day" in df_day.columns and df_day["sim_day"].notna().any():
    df_day["day"] = df_day["sim_day"]
    day_basis = "jour simulé (sim_day)"
else:
    df_day["day"] = df_day["time"].dt.strftime("%Y-%m-%d")
    day_basis = "horloge murale — sim_ts absent (relancer une simulation pour l'axe jour simulé)"

# ── Agents : agent_id parfois mal formé par le LLM ("PERSONA 446264", un nom…) ──
# On normalise sur la partie numérique pour ne compter qu'une fois chaque agent réel.
raw_ids = set()
for r in df_day.loc[df_day["category"] == "itinary_multi_agent", "response"]:
    if isinstance(r, list):
        raw_ids.update(str(a["agent_id"]) for a in r if isinstance(a, dict) and "agent_id" in a)

def _norm_id(i):
    m = re.search(r"\d+", i)          # extrait le numéro d'agent
    return m.group() if m else None

valid_ids  = sorted({_norm_id(i) for i in raw_ids if _norm_id(i)})
anomalies  = sorted(i for i in raw_ids if not i.isdigit())   # ids non purement numériques
n_agents   = len(valid_ids)

print(f"agent_id distincts (bruts) : {len(raw_ids)}  →  agents réels (normalisés) : {n_agents}")
print(f"  ids réels : {valid_ids}")
if anomalies:
    print(f"  ⚠️ {len(anomalies)} agent_id mal formés par le LLM (normalisés sur le numéro) : {anomalies}")

# Tokens par jour et catégorie → puis CUMUL depuis le début (chaque jour = total accumulé)
by_day = (df_day.groupby(["day", "category"])["tokens_total"].sum()
          .unstack("category", fill_value=0).sort_index())
cum = by_day.cumsum()

# ── Économie estimée du cache (llm_cache_hits.jsonl, si présent), CUMULÉE ──────
saved_cum, n_hits = None, 0
cache_path = os.path.join(LOG_DIR, "llm_cache_hits.jsonl")
if os.path.exists(cache_path):
    hits = [json.loads(l) for l in open(cache_path, encoding="utf-8") if l.strip()]
    n_hits = len(hits)
    if hits:
        dfh = pd.DataFrame(hits)
        it = df_day[df_day["category"] == "itinary_multi_agent"]
        n_ag = it["response"].apply(lambda r: max(len(r), 1) if isinstance(r, list) else 1).sum()
        mean_tok_per_agent = it["tokens_total"].sum() / n_ag if n_ag else 0
        day_col = "sim_day" if "sim_day" in dfh.columns and dfh["sim_day"].notna().any() else "time"
        s = dfh.groupby(dfh[day_col].astype(str).str[:10]).size() * mean_tok_per_agent
        saved_cum = s.reindex(by_day.index, fill_value=0).cumsum()  # cumul depuis le début

# ── Graphe : conso cumulée empilée par catégorie vs budget journalier constant ─
days = list(cum.index)
x = np.arange(len(days))
fig, ax = plt.subplots(figsize=(14, 7))

bottom = np.zeros(len(days))  # base de la pile à 0 (échelle linéaire standard)
colors = plt.cm.tab10(np.linspace(0, 1, len(cum.columns)))
for cat, color in zip(cum.columns, colors):
    top = bottom + cum[cat].values
    ax.fill_between(x, bottom, top, color=color, alpha=0.75, label=cat)
    ax.plot(x, top, color=color, lw=1.5)
    bottom = top

total_cum = bottom  # sommet de la pile = total cumulé depuis le début
ax.plot(x, total_cum, color="black", lw=2.2, label="TOTAL cumulé")

if saved_cum is not None:
    ax.plot(x, total_cum + saved_cum.values, color="grey", lw=1.8, ls=":",
            label="TOTAL sans cache cumulé (est.)")

# Budget journalier = CONSTANT (ligne plate) — commenté (écraserait la conso en linéaire)
# ax.axhline(DAILY_TOKEN_LIMIT, color="red", ls="--", lw=2,
#            label=f"budget {DAILY_TOKEN_LIMIT/1e6:.0f} M/jour (constant)")

# Échelle linéaire standard, axe Y libellé en MILLIONS de tokens
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1e6:g} M"))
ax.set_xticks(x)
ax.set_xticklabels(days, rotation=45, ha="right")
ax.set_title(f"Consommation cumulée de tokens par catégorie — {n_agents} agents (M de tokens)\n(base : {day_basis})",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Jour")
ax.set_ylabel("Tokens cumulés (M)")
ax.legend(loc="upper left", fontsize=8, ncol=2)
ax.grid(axis="y", which="both", alpha=0.3)
ax.set_xlim(0, len(days) - 1)

plt.tight_layout()
plt.savefig(_img_path("tokens_cumul_vs_budget"), dpi=200)
plt.show()

# ── Récap en M de tokens ──────────────────────────────────────────────────────
final_total = float(total_cum[-1])
print()
print(f"Agents réels                 : {n_agents}")
print(f"Budget journalier (constant) : {DAILY_TOKEN_LIMIT/1e6:,.1f} M tokens/jour")
print(f"Total cumulé à J{len(days)}        : {final_total/1e6:.4f} M tokens "
      f"({final_total/DAILY_TOKEN_LIMIT*100:.4f} % d'un budget journalier)")
if saved_cum is not None:
    print(f"Cache : {n_hits} hits → ~{saved_cum.iloc[-1]/1e6:.4f} M tokens cumulés économisés")


## Consommation cumulée de tokens vs budget — V2 interactive (Plotly)

Même donnée que ci-dessus, mais en **graphe interactif** : zoom à la molette / sélection rectangle, pan, survol des valeurs exactes (tooltip), **range slider** sous l'axe X, boutons de plage rapide, et **légende cliquable** pour isoler/masquer une catégorie.

> Réutilise les variables calculées dans la cellule précédente (`cum`, `saved_cum`, `DAILY_TOKEN_LIMIT`, `n_agents`, `day_basis`). Exécute d'abord la V1, puis cette cellule.
>
> Astuce : double-clic sur le graphe pour réinitialiser le zoom ; outils zoom/pan/box-select dans la barre en haut à droite.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np

# Réutilise cum / saved_cum / DAILY_TOKEN_LIMIT / n_agents / day_basis de la cellule V1.
days_v2 = [str(d) for d in cum.index]
fig = go.Figure()

# ── Aires empilées par catégorie (légende cliquable pour isoler/masquer) ──────
palette = pio.templates["plotly"].layout.colorway or [
    "#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A",
    "#19D3F3", "#FF6692", "#B6E880", "#FF97FF", "#FECB52",
]
for i, cat in enumerate(cum.columns):
    fig.add_trace(go.Scatter(
        x=days_v2, y=cum[cat].values,
        name=str(cat), mode="lines",
        line=dict(width=1.2, color=palette[i % len(palette)]),
        stackgroup="conso", groupnorm="",
        hovertemplate=f"<b>{cat}</b><br>%{{x}}<br>%{{y:,.0f}} tokens<extra></extra>",
    ))

total_cum_v2 = cum.sum(axis=1).values

# ── Ligne TOTAL cumulé (hors pile) ────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=days_v2, y=total_cum_v2, name="TOTAL cumulé", mode="lines",
    line=dict(color="black", width=2.6),
    hovertemplate="<b>TOTAL</b><br>%{x}<br>%{y:,.0f} tokens<extra></extra>",
))

# ── Ligne TOTAL sans cache (si dispo) ─────────────────────────────────────────
if saved_cum is not None:
    fig.add_trace(go.Scatter(
        x=days_v2, y=total_cum_v2 + saved_cum.values,
        name="TOTAL sans cache (est.)", mode="lines",
        line=dict(color="grey", width=2, dash="dot"),
        hovertemplate="<b>sans cache</b><br>%{x}<br>%{y:,.0f} tokens<extra></extra>",
    ))

fig.update_layout(
    title=dict(
        text=f"Consommation cumulée de tokens par catégorie — {n_agents} agents"
             f"<br><sup>base : {day_basis}</sup>",
        font=dict(size=15),
    ),
    height=620,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(l=70, r=30, t=90, b=40),
    dragmode="zoom",  # box-zoom par défaut ; box-select dispo dans la modebar
)
fig.update_xaxes(
    title_text="Jour",
    rangeslider=dict(visible=True),  # mini-carte + sélection de plage sous l'axe
    rangeselector=dict(buttons=[
        dict(count=7, label="7j", step="day", stepmode="backward"),
        dict(count=1, label="1m", step="month", stepmode="backward"),
        dict(step="all", label="tout"),
    ]) if np.issubdtype(np.asarray(days_v2).dtype, np.datetime64) else None,
)
fig.update_yaxes(
    title_text="Tokens cumulés",
    tickformat="~s",  # 1.2M, 340k…  -> survol pour la valeur exacte
)

# Modebar : zoom / pan / box-select / lasso / reset toujours visibles
config = {
    "scrollZoom": True,
    "displaylogo": False,
    "modeBarButtonsToAdd": ["select2d", "lasso2d", "drawrect", "eraseshape"],
    "toImageButtonOptions": {"filename": "tokens_cumul_vs_budget_v2", "scale": 2},
}

# Sauvegarde HTML interactif (à côté des PNG des autres graphes)
html_path = _img_path("tokens_cumul_vs_budget_v2", ext="html")
fig.write_html(html_path, include_plotlyjs="cdn", config=config)
print(f"HTML interactif sauvegardé : {html_path}")

fig.show(config=config)


In [ ]:
token_mensuel_10_agents = 2310230
token_mensuel_1_agents = token_mensuel_10_agents / 10
max_free_plan_tokens = 338540000  # Limite gratuite mensuelle pour un seul agent
moy_act = 4.5

nb_jours_1_agent = max_free_plan_tokens / token_mensuel_1_agents
print(f"Nombre de jours estimé pour 1 agent avec le plan gratuit : {nb_jours_1_agent:.2f} jours")
print(f"Nombre de mois estimé pour 1 agent avec le plan gratuit : {nb_jours_1_agent/30:.2f} mois")

nb_act_1_agent = nb_jours_1_agent*moy_act
print(f"Nombre d'activité avec les plans gratuits : {nb_act_1_agent}")


